# Code to generate Figure 2 subplots

In [1]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from scipy.stats import mannwhitneyu
from tqdm import trange

DATA_PATH1 = "../../0-download-inputs/data-files/"
DATA_PATH2 = "../../flatten/processed-data/"
DATA_PATH3 = "../../18-Y2H-data/processed-data/"
ABUNDANCE  = "../../0-download-inputs/data-files/1-s2.0-S240547121730546X-mmc5.csv" # required to join with Y2H dataset
OUT_PATH   = "figures/"
MS_COLOR   = "#DDAA33"
Y2H_COLOR  = "#BB5566"

plt.rcParams.update({
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "axes.titlesize": 14,
    "axes.titlepad": 25,
    "font.family": "sans-serif",
    "font.sans-serif": "Arial",
    "figure.facecolor": "white",
    "axes.facecolor": "white"
})

## Figure 2a
* Create boxplots of abundance for hubs and non-hubs in mass spec and Y2H datasets
* panel a will be hubs based on degree centrality, and panel b will be hubs based on betweenness centrality

In [2]:
# load the datasets with node centrality information and get nodes in both
ms_nodes = pd.read_pickle(f"{DATA_PATH2}20251209-s288c-annotated-PPI-net.pkl")
y2h_nodes = pd.read_csv(f"{DATA_PATH3}Y2H-s288c-step18.csv")

# while the MS data already contains mean_molecules_per_cell and median_molecules_per_cell columns derived from 
# 10.1016/j.cels.2017.12.004, we need to merge this information into the y2h_nodes network
abundance_df = (
    pd.read_csv(
        ABUNDANCE,
        usecols=[
            "Systematic Name",
            "Mean molecules per cell",
            "Median molecules per cell",
        ],
    )
    .rename(
        columns={
            "Systematic Name": "node",
            "Mean molecules per cell": "mean_molecules_per_cell",
            "Median molecules per cell": "median_molecules_per_cell",
        }
    )
)
y2h_nodes = y2h_nodes.merge(abundance_df, on="node", how="left")

shared_nodes = set(ms_nodes["node"]) & set(y2h_nodes["node"])
print (f"A total of {len(shared_nodes)} nodes are shared between the MS and Y2H networks")

FileNotFoundError: [Errno 2] No such file or directory: '../../0-download-inputs/data-files/1-s2.0-S240547121730546X-mmc5.csv'

In [ ]:
# determine which nodes are hubs based on percentile cutoff for degree centrality
cut = 0.10
ms_threshold = ms_nodes["degree_centrality"].quantile(1 - cut)
ms_hubs = ms_nodes[ms_nodes["degree_centrality"] >= ms_threshold]
ms_nothub  = ms_nodes[ms_nodes["degree_centrality"] <  ms_threshold]

y2h_threshold = y2h_nodes["degree_centrality"].quantile(1 - cut)
y2h_hubs   = y2h_nodes[y2h_nodes["degree_centrality"] >= y2h_threshold]
y2h_nothub = y2h_nodes[y2h_nodes["degree_centrality"] <  y2h_threshold]

print (f"Threshold for defining a hub in MS data is {ms_threshold:.5f}; there are {len(ms_hubs)} hubs and {len(ms_nothub)} non hubs")
print (f"Threshold for defining a hub in Y2H datais {y2h_threshold:.5f}; there are {len(y2h_hubs)} hubs and {len(y2h_nothub)} non hubs")

Compute medians in each group (hubs and non-hubs) within each dataset along with bootstrapped 95% confidence intervals

In [ ]:
def bootstrap_ci_median(values, B=1_000_000, ci=95, random_state=27):
    """
    Nonparametric bootstrap CI for the median.

    values: 1D array-like
    B: number of bootstrap resamples
    ci: confidence level (e.g., 95)
    """
    vals = np.asarray(values)
    vals = vals[np.isfinite(vals)]
    n = len(vals)
    if n == 0:
        return np.nan, np.nan, np.nan

    rng = np.random.default_rng(random_state)
    medians = np.empty(B, dtype=float)

    # Loop-based bootstrap to avoid huge memory allocations
    for b in trange(B, desc="Bootstrapping medians"):
        sample = rng.choice(vals, size=n, replace=True)
        medians[b] = np.median(sample)

    median_hat = np.median(vals)
    alpha = 100 - ci
    lower = np.percentile(medians, alpha / 2)
    upper = np.percentile(medians, 100 - alpha / 2)
    return median_hat, lower, upper

def clean_for_log(s):
    return (
        s.replace([np.inf, -np.inf], np.nan)
         .dropna()
         .loc[lambda x: x > 0]
    )

ms_nothub_vals  = clean_for_log(ms_nothub["mean_molecules_per_cell"])
ms_hub_vals     = clean_for_log(ms_hubs["mean_molecules_per_cell"])
y2h_nothub_vals = clean_for_log(y2h_nothub["mean_molecules_per_cell"])
y2h_hub_vals    = clean_for_log(y2h_hubs["mean_molecules_per_cell"])

groups = [
    ("MS hubs",      ms_hub_vals,     MS_COLOR),
    ("MS non-hubs",  ms_nothub_vals,  MS_COLOR),
    ("Y2H hubs",     y2h_hub_vals,    Y2H_COLOR),
    ("Y2H non-hubs", y2h_nothub_vals, Y2H_COLOR),
]

medians = []
ci_lowers = []
ci_uppers = []
labels = []
colors = []

for label, vals, color in groups:
    m, lo, hi = bootstrap_ci_median(vals, B=1_000_000, ci=95, random_state=42)
    medians.append(m)
    ci_lowers.append(lo)
    ci_uppers.append(hi)
    labels.append(label)
    colors.append(color)
    print(f"{label}: median={m:.2f}, 95% CI=({lo:.2f}, {hi:.2f}), n={len(vals)})")

medians = np.array(medians)
ci_lowers = np.array(ci_lowers)
ci_uppers = np.array(ci_uppers)

# Convert to yerr format for Matplotlib: [lower_err, upper_err]
yerr = np.vstack([medians - ci_lowers, ci_uppers - medians])

The average median molecules per cell within MS hubs is 9306 [7726, 11208] (95% confidence intervals computed with non-parametric bootstrap using 1,000,000 random samples). 

In [ ]:
# MS: hubs vs non-hubs
ms_u, ms_p = mannwhitneyu(ms_hub_vals, ms_nothub_vals, alternative="two-sided")

# Y2H: hubs vs non-hubs
y2h_u, y2h_p = mannwhitneyu(y2h_hub_vals, y2h_nothub_vals, alternative="two-sided")

print("=== Mann–Whitney U tests on mean_molecules_per_cell (raw values) ===")

print("\nMS hubs vs MS non-hubs")
print(f"n(hubs) = {len(ms_hub_vals)}, n(non-hubs) = {len(ms_nothub_vals)}")
print(f"median(hubs) = {np.median(ms_hub_vals):.2f}, median(non-hubs) = {np.median(ms_nothub_vals):.2f}")
print(f"U = {ms_u:.2f}, p = {ms_p:.4g}")

print("\nY2H hubs vs Y2H non-hubs")
print(f"n(hubs) = {len(y2h_hub_vals)}, n(non-hubs) = {len(y2h_nothub_vals)}")
print(f"median(hubs) = {np.median(y2h_hub_vals):.2f}, median(non-hubs) = {np.median(y2h_nothub_vals):.2f}")
print(f"U = {y2h_u:.2f}, p = {y2h_p:.4g}")

In [ ]:
# Plot annotations
def add_bracket_with_text(ax, x1, x2, y, text):
    y_top = y * 1.1
    ax.plot(
        [x1, x1, x2, x2],
        [y,  y_top, y_top, y],
        lw=1.5,
        c="black",
        zorder=4,
    )
    ax.text(
        (x1 + x2) / 2,
        y_top * 1.05,
        text,
        ha="center",
        va="bottom",
        fontsize=11,
        zorder=4,
    )

fig, ax = plt.subplots(figsize=(3, 3))

group_vals   = [ms_hub_vals, ms_nothub_vals, y2h_hub_vals, y2h_nothub_vals]
group_colors = [MS_COLOR,    MS_COLOR,       Y2H_COLOR,    Y2H_COLOR]

# --- Scatter first (lower zorder) ---
for i, (vals, color) in enumerate(zip(group_vals, group_colors), start=1):
    x_jitter = i + np.random.normal(0, 0.05, size=len(vals))
    ax.scatter(
        x_jitter,
        vals,
        s=10,
        alpha=0.8,
        color=color,
        edgecolor="none",
        zorder=2,
    )

# --- Boxplots on top: outlines only, no fill, no fliers ---
bp = ax.boxplot(
    group_vals,
    labels=["Hubs", "Non-\nhubs", "Hubs", "Non-\nhubs"],
    patch_artist=True,
    showfliers=False,
)

for box in bp["boxes"]:
    box.set_facecolor("none")
    box.set_edgecolor("black")
    box.set_linewidth(1.5)
    box.set_zorder(3)

for element in ["whiskers", "caps", "medians"]:
    for line in bp[element]:
        line.set_color("black")
        line.set_linewidth(1.2)
        line.set_zorder(3)

# --- Axes styling ---
ax.set_yscale("log")
ax.set_ylim(1, 1e7)

ax.set_ylabel("Molecular abundance")
#ax.set_title("Degree Centrality")

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_linewidth(1.5)
ax.spines["bottom"].set_linewidth(1.5)

ms_pair_max  = max(ms_hub_vals.max(), ms_nothub_vals.max())
y2h_pair_max = max(y2h_hub_vals.max(), y2h_nothub_vals.max())

ms_y  = ms_pair_max * 2.0
y2h_y = y2h_pair_max * 2.0

add_bracket_with_text(ax, 1, 2, ms_y,  "***")
add_bracket_with_text(ax, 3, 4, y2h_y, "N.S.")

# === Horizontal lines + labels under the groups ===
# Use x-axis transform so these sit below the x-axis, not in data (log) space
trans = ax.get_xaxis_transform()  # x in data coords, y in axis fraction (0–1)

# AE-MS under boxes 1–2
ax.plot(
    [0.9, 2.1], [-0.29, -0.29],
    transform=trans,
    clip_on=False,
    color=MS_COLOR,
    lw=1.5,
)
ax.text(
    1.5, -0.31, "AE-MS",
    transform=trans,
    ha="center",
    va="top",
    fontsize=10,
    fontweight="bold",
    color=MS_COLOR,
)

# Y2H under boxes 3–4
ax.plot(
    [2.9, 4.1], [-0.29, -0.29],
    transform=trans,
    clip_on=False,
    color=Y2H_COLOR,
    lw=1.5,
)
ax.text(
    3.5, -0.31, "Y2H",
    transform=trans,
    ha="center",
    va="top",
    fontsize=10,
    fontweight="bold",
    color=Y2H_COLOR,
)
plt.xticks(fontsize=10)
plt.yticks([1, 10, 100, 1000, 10000, 100000, 100000, 1000000, 10000000])
plt.tight_layout()
plt.savefig(f"{OUT_PATH}2a.svg")
plt.show()

## Figure 2b

Do the same thing again but for the betweenness centrality

In [ ]:
# determine which nodes are hubs based on percentile cutoff for degree centrality
cut = 0.10
ms_threshold = ms_nodes["betweenness_centrality"].quantile(1 - cut)
ms_hubs = ms_nodes[ms_nodes["betweenness_centrality"] >= ms_threshold]
ms_nothub  = ms_nodes[ms_nodes["betweenness_centrality"] <  ms_threshold]

y2h_threshold = y2h_nodes["betweenness_centrality"].quantile(1 - cut)
y2h_hubs   = y2h_nodes[y2h_nodes["betweenness_centrality"] >= y2h_threshold]
y2h_nothub = y2h_nodes[y2h_nodes["betweenness_centrality"] <  y2h_threshold]

print (f"Threshold for defining a hub in MS data is {ms_threshold:.5f}; there are {len(ms_hubs)} hubs and {len(ms_nothub)} non hubs")
print (f"Threshold for defining a hub in Y2H datais {y2h_threshold:.5f}; there are {len(y2h_hubs)} hubs and {len(y2h_nothub)} non hubs")

In [ ]:
ms_nothub_vals  = clean_for_log(ms_nothub["mean_molecules_per_cell"])
ms_hub_vals     = clean_for_log(ms_hubs["mean_molecules_per_cell"])
y2h_nothub_vals = clean_for_log(y2h_nothub["mean_molecules_per_cell"])
y2h_hub_vals    = clean_for_log(y2h_hubs["mean_molecules_per_cell"])

groups = [
    ("MS hubs",      ms_hub_vals,     MS_COLOR),
    ("MS non-hubs",  ms_nothub_vals,  MS_COLOR),
    ("Y2H hubs",     y2h_hub_vals,    Y2H_COLOR),
    ("Y2H non-hubs", y2h_nothub_vals, Y2H_COLOR),
]

medians = []
ci_lowers = []
ci_uppers = []
labels = []
colors = []

for label, vals, color in groups:
    m, lo, hi = bootstrap_ci_median(vals, B=1_000_000, ci=95, random_state=42)
    medians.append(m)
    ci_lowers.append(lo)
    ci_uppers.append(hi)
    labels.append(label)
    colors.append(color)
    print(f"{label}: median={m:.2f}, 95% CI=({lo:.2f}, {hi:.2f}), n={len(vals)})")

medians = np.array(medians)
ci_lowers = np.array(ci_lowers)
ci_uppers = np.array(ci_uppers)

# Convert to yerr format for Matplotlib: [lower_err, upper_err]
yerr = np.vstack([medians - ci_lowers, ci_uppers - medians])

In [ ]:
# MS: hubs vs non-hubs
ms_u, ms_p = mannwhitneyu(ms_hub_vals, ms_nothub_vals, alternative="two-sided")

# Y2H: hubs vs non-hubs
y2h_u, y2h_p = mannwhitneyu(y2h_hub_vals, y2h_nothub_vals, alternative="two-sided")

print("=== Mann–Whitney U tests on mean_molecules_per_cell (raw values) ===")

print("\nMS bottlenecks vs MS non-bottlenecks")
print(f"n(hubs) = {len(ms_hub_vals)}, n(non-hubs) = {len(ms_nothub_vals)}")
print(f"median(hubs) = {np.median(ms_hub_vals):.2f}, median(non-hubs) = {np.median(ms_nothub_vals):.2f}")
print(f"U = {ms_u:.2f}, p = {ms_p:.4g}")

print("\nY2H bottlenecks vs Y2H bottlenecks")
print(f"n(hubs) = {len(y2h_hub_vals)}, n(non-hubs) = {len(y2h_nothub_vals)}")
print(f"median(hubs) = {np.median(y2h_hub_vals):.2f}, median(non-hubs) = {np.median(y2h_nothub_vals):.2f}")
print(f"U = {y2h_u:.2f}, p = {y2h_p:.4g}")

In [ ]:
fig, ax = plt.subplots(figsize=(3, 3))

group_vals   = [ms_hub_vals, ms_nothub_vals, y2h_hub_vals, y2h_nothub_vals]
group_colors = [MS_COLOR,    MS_COLOR,       Y2H_COLOR,    Y2H_COLOR]

# --- Scatter first (lower zorder) ---
for i, (vals, color) in enumerate(zip(group_vals, group_colors), start=1):
    x_jitter = i + np.random.normal(0, 0.05, size=len(vals))
    ax.scatter(
        x_jitter,
        vals,
        s=10,
        alpha=0.8,
        color=color,
        edgecolor="none",
        zorder=2,
    )

# --- Boxplots on top: outlines only, no fill, no fliers ---
bp = ax.boxplot(
    group_vals,
    labels=["Bottlenecks", "Non-\nbottlenecks", "Bottlenecks", "Non-\nbottlenecks"],
    patch_artist=True,
    showfliers=False,
)

for box in bp["boxes"]:
    box.set_facecolor("none")
    box.set_edgecolor("black")
    box.set_linewidth(1.5)
    box.set_zorder(3)

for element in ["whiskers", "caps", "medians"]:
    for line in bp[element]:
        line.set_color("black")
        line.set_linewidth(1.2)
        line.set_zorder(3)

# --- Axes styling ---
ax.set_yscale("log")
ax.set_ylim(1, 1e7)

ax.set_ylabel("Molecular abundance")
#ax.set_title("Betweenness Centrality")

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_linewidth(1.5)
ax.spines["bottom"].set_linewidth(1.5)

ms_pair_max  = max(ms_hub_vals.max(), ms_nothub_vals.max())
y2h_pair_max = max(y2h_hub_vals.max(), y2h_nothub_vals.max())

ms_y  = ms_pair_max * 2.0
y2h_y = y2h_pair_max * 2.0

add_bracket_with_text(ax, 1, 2, ms_y,  "***")
add_bracket_with_text(ax, 3, 4, y2h_y, "N.S.")

# === Horizontal lines + labels under the groups ===
# Use x-axis transform so these sit below the x-axis, not in data (log) space
trans = ax.get_xaxis_transform()  # x in data coords, y in axis fraction (0–1)

# AE-MS under boxes 1–2
ax.plot(
    [0.9, 2.1], [-0.29, -0.29],
    transform=trans,
    clip_on=False,
    color=MS_COLOR,
    lw=1.5,
)
ax.text(
    1.5, -0.31, "AE-MS",
    transform=trans,
    ha="center",
    va="top",
    fontsize=10,
    fontweight="bold",
    color=MS_COLOR,
)

# Y2H under boxes 3–4
ax.plot(
    [2.9, 4.1], [-0.29, -0.29],
    transform=trans,
    clip_on=False,
    color=Y2H_COLOR,
    lw=1.5,
)
ax.text(
    3.5, -0.31, "Y2H",
    transform=trans,
    ha="center",
    va="top",
    fontsize=10,
    fontweight="bold",
    color=Y2H_COLOR,
)
ax.tick_params(axis="x", labelsize=8)
for label in ax.get_xticklabels():
    label.set_rotation(-30)   # or 45
    label.set_ha("center")
    label.set_fontsize(6.5)
plt.yticks([1, 10, 100, 1000, 10000, 100000, 100000, 1000000, 10000000])
plt.savefig(f"{OUT_PATH}2b.svg")
plt.show()